# Project · A tiny Transformer and self-attention from scratch

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). Write a working self-attention block in pure NumPy—sinusoidal positional encodings, scaled dot-product attention forward and backward, a finite-difference gradient check, and training on an associative key-value lookup task where a position-blind bag-of-words model fails completely. Uses Murphy 1 ch. 15, D2L ch. 11 and Murphy 2 ch. 22.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · An associative lookup task that needs attention

Create sequences of 5 tokens: two key-value pairs (k1, v1, k2, v2) followed by a query key q ∈ {k1, k2}, where keys are in {0..3} and values are in {4..7}. The target is the value paired with q. Generate 1,200 training and 300 test sequences.

In [ ]:
# Your turn
import numpy as np
rng = np.random.default_rng(0)
def make_data(N):
    X = np.zeros((N, 5), int); y = np.zeros(N, int)
    for n in range(N):
        k1, k2 = rng.choice(4, 2, replace=False)
        v1, v2 = rng.choice(4, 2, replace=False) + 4
        pick = rng.integers(0, 2)
        q = k1 if pick == 0 else k2
        X[n] = [k1, v1, k2, v2, q]
        y[n] = ...  # TODO: target class in 0..3
    return X, y
Xtr, ytr = make_data(1200); Xte, yte = make_data(300)
print("Xtr:", Xtr.shape, "sample sequence:", Xtr[0].tolist(), "-> target class:", int(ytr[0]))

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
rng = np.random.default_rng(0)
def make_data(N):
    X = np.zeros((N, 5), int); y = np.zeros(N, int)
    for n in range(N):
        k1, k2 = rng.choice(4, 2, replace=False)
        v1, v2 = rng.choice(4, 2, replace=False) + 4
        pick = rng.integers(0, 2)
        q = k1 if pick == 0 else k2
        X[n] = [k1, v1, k2, v2, q]
        y[n] = (v1 if pick == 0 else v2) - 4  # target class in 0..3
    return X, y
Xtr, ytr = make_data(1200); Xte, yte = make_data(300)
print("Xtr:", Xtr.shape, "sample sequence:", Xtr[0].tolist(), "-> target class:", int(ytr[0]))

<details><summary><b>Answer and explanation</b></summary>

Each sequence [k1, v1, k2, v2, q] contains the exact same bag of tokens whether q = k1 or q = k2 (only the last token changes, and the answer depends on which earlier position matches q and reading its right-hand neighbour). A bag-of-words model that averages token embeddings cannot tell which value belongs to which key; an attention head that looks from position 4 (q) to position 0 or 2 and reads the adjacent value solves it cleanly.

Expected output:
```
Xtr: (1200, 5) sample sequence: [2, 5, 3, 4, 2] -> target class: 1
```

</details>

## Milestone 2 · Token embeddings + sinusoidal positional encoding

Build a sinusoidal positional table PE[t, 2i] = sin(t / 100^((2i)/d)), PE[t, 2i+1] = cos(t / 100^((2i)/d)) for length T=5 and d=16. Verify that distinct positions have distinct vectors and check the dot products with position 0.

In [ ]:
# Your turn
T, D, V = 5, 16, 8
pos = np.arange(T)[:, None]; div = 100.0 ** (np.arange(0, D, 2) / D)
PE = np.zeros((T, D))
PE[:, 0::2] = np.sin(pos / div); PE[:, 1::2] = ...  # TODO: sinusoidal position table
dots = (PE @ PE[0]).round(3)
print("PE shape:", PE.shape, "dot products with t=0:", dots.tolist())

In [ ]:
# @title Solution 2 (double-click to show the code)
T, D, V = 5, 16, 8
pos = np.arange(T)[:, None]; div = 100.0 ** (np.arange(0, D, 2) / D)
PE = np.zeros((T, D))
PE[:, 0::2] = np.sin(pos / div); PE[:, 1::2] = np.cos(pos / div)  # sinusoidal position table
dots = (PE @ PE[0]).round(3)
print("PE shape:", PE.shape, "dot products with t=0:", dots.tolist())

<details><summary><b>Answer and explanation</b></summary>

Without positional encodings, scaled dot-product attention is permutation-equivariant: swapping (k1, v1) into (k1, v2) and (k2, v1) would look identical! Adding PE[t] stamps every slot t = 0..4 with a unique phase signature so key at t can bind to value at t+1 (Murphy 1 §15.4.5, D2L §11.6).

Expected output:
```
PE shape: (5, 16) dot products with t=0: [8.0, 7.314, 5.73, 4.273, 3.663]
```

</details>

## Milestone 3 · Scaled dot-product attention: forward, backward and gradient check

Write a single-layer attention classifier: embed X -> H = E[X] + PE, compute Q = H W_Q, K = H W_K, Vv = H W_V, attention A = softmax(Q K^T / sqrt(D)), take the query token's attended output c = A[:, -1, :] @ Vv, and project logits = c @ W_O + b. Derive the exact analytic backward pass and verify against finite differences.

In [ ]:
# Your turn
def init_attn(seed=1):
    r = np.random.default_rng(seed)
    s = 0.25
    return {"E": r.normal(0, s, (V, D)), "WQ": r.normal(0, s, (D, D)), "WK": r.normal(0, s, (D, D)),
            "WV": r.normal(0, s, (D, D)), "WO": r.normal(0, s, (D, 4)), "b": np.zeros(4)}
def attn_fwd(P, X):
    H = ...  # TODO: (N, T, D) token + pos
    Q = H @ P["WQ"]; K = H @ P["WK"]; Vv = H @ P["WV"]
    sc = ...  # TODO: query t=4 attends to t=0..4
    sc -= sc.max(1, keepdims=True); A = np.exp(sc); A /= A.sum(1, keepdims=True)
    c = ...  # TODO: attended context (N, D)
    logits = c @ P["WO"] + P["b"]
    logits -= logits.max(1, keepdims=True); p = np.exp(logits); p /= p.sum(1, keepdims=True)
    return p, (X, H, Q, K, Vv, A, c)
def attn_bwd(P, Y, cache, p):
    X, H, Q, K, Vv, A, c = cache; N = len(X)
    dl = (p - Y) / N
    g = {"WO": c.T @ dl, "b": dl.sum(0)}
    dc = ...  # TODO: (N, D)
    dA = ...  # TODO: (N, T)
    dVv = ...  # TODO: (N, T, D)
    dsc = ...  # TODO: softmax Jacobian / sqrt(D)
    dQ_last = ...  # TODO: (N, D)
    dK = ...  # TODO: (N, T, D)
    g["WQ"] = H[:, -1, :].T @ dQ_last
    g["WK"] = (H.transpose(0, 2, 1) @ dK).sum(0)
    g["WV"] = (H.transpose(0, 2, 1) @ dVv).sum(0)
    dH = dK @ P["WK"].T + dVv @ P["WV"].T
    dH[:, -1, :] += dQ_last @ P["WQ"].T
    g["E"] = np.zeros_like(P["E"]); np.add.at(g["E"], X, dH)
    return g
P = init_attn(); Xb, yb = Xtr[:32], ytr[:32]; Yb = np.eye(4)[yb]
p, cache = attn_fwd(P, Xb); g = attn_bwd(P, Yb, cache, p)
loss = lambda P: -np.mean(np.log(attn_fwd(P, Xb)[0][np.arange(len(yb)), yb]))
for k in ["E", "WQ", "WK", "WV", "WO"]:
    idx = (0, 0); h = 1e-5
    P[k][idx] += h; lp = loss(P); P[k][idx] -= 2 * h; lm = loss(P); P[k][idx] += h
    print(f"{k:>2}: analytic {g[k][idx]: .7f}  numeric {(lp - lm) / (2 * h): .7f}")

In [ ]:
# @title Solution 3 (double-click to show the code)
def init_attn(seed=1):
    r = np.random.default_rng(seed)
    s = 0.25
    return {"E": r.normal(0, s, (V, D)), "WQ": r.normal(0, s, (D, D)), "WK": r.normal(0, s, (D, D)),
            "WV": r.normal(0, s, (D, D)), "WO": r.normal(0, s, (D, 4)), "b": np.zeros(4)}
def attn_fwd(P, X):
    H = P["E"][X] + PE  # (N, T, D) token + pos
    Q = H @ P["WQ"]; K = H @ P["WK"]; Vv = H @ P["WV"]
    sc = (Q[:, -1:, :] @ K.transpose(0, 2, 1))[:, 0, :] / np.sqrt(D)  # query t=4 attends to t=0..4
    sc -= sc.max(1, keepdims=True); A = np.exp(sc); A /= A.sum(1, keepdims=True)
    c = (A[:, :, None] * Vv).sum(1)  # attended context (N, D)
    logits = c @ P["WO"] + P["b"]
    logits -= logits.max(1, keepdims=True); p = np.exp(logits); p /= p.sum(1, keepdims=True)
    return p, (X, H, Q, K, Vv, A, c)
def attn_bwd(P, Y, cache, p):
    X, H, Q, K, Vv, A, c = cache; N = len(X)
    dl = (p - Y) / N
    g = {"WO": c.T @ dl, "b": dl.sum(0)}
    dc = dl @ P["WO"].T  # (N, D)
    dA = (dc[:, None, :] * Vv).sum(-1)  # (N, T)
    dVv = A[:, :, None] * dc[:, None, :]  # (N, T, D)
    dsc = A * (dA - (dA * A).sum(1, keepdims=True)) / np.sqrt(D)  # softmax Jacobian / sqrt(D)
    dQ_last = (dsc[:, :, None] * K).sum(1)  # (N, D)
    dK = dsc[:, :, None] * Q[:, -1:, :]  # (N, T, D)
    g["WQ"] = H[:, -1, :].T @ dQ_last
    g["WK"] = (H.transpose(0, 2, 1) @ dK).sum(0)
    g["WV"] = (H.transpose(0, 2, 1) @ dVv).sum(0)
    dH = dK @ P["WK"].T + dVv @ P["WV"].T
    dH[:, -1, :] += dQ_last @ P["WQ"].T
    g["E"] = np.zeros_like(P["E"]); np.add.at(g["E"], X, dH)
    return g
P = init_attn(); Xb, yb = Xtr[:32], ytr[:32]; Yb = np.eye(4)[yb]
p, cache = attn_fwd(P, Xb); g = attn_bwd(P, Yb, cache, p)
loss = lambda P: -np.mean(np.log(attn_fwd(P, Xb)[0][np.arange(len(yb)), yb]))
for k in ["E", "WQ", "WK", "WV", "WO"]:
    idx = (0, 0); h = 1e-5
    P[k][idx] += h; lp = loss(P); P[k][idx] -= 2 * h; lm = loss(P); P[k][idx] += h
    print(f"{k:>2}: analytic {g[k][idx]: .7f}  numeric {(lp - lm) / (2 * h): .7f}")

<details><summary><b>Answer and explanation</b></summary>

Every analytic gradient (E, W_Q, W_K, W_V, W_O) agrees with centered finite differences to 1e-7. Notice the softmax Jacobian inside backward: dS = A * (dA - sum(dA * A, axis=-1, keepdims=True)) / sqrt(D). If an attention weight A_ij is near 0 or 1, its gradient dS_ij vanishes—which is exactly why dividing by sqrt(D) is essential before softmax!

Expected output:
```
 E: analytic -0.0073083  numeric -0.0073083
WQ: analytic -0.0002836  numeric -0.0002836
WK: analytic -0.0007236  numeric -0.0007236
WV: analytic -0.0222519  numeric -0.0222519
WO: analytic  0.0439009  numeric  0.0439009
```

</details>

## Milestone 4 · Train the attention block vs a bag-of-words baseline

Train the attention model with Adam (lr=0.02, 40 epochs, batch 64) and compare its test accuracy against a position-aware linear probe on mean-pooled embeddings. Inspect the learned attention weights on a test example.

In [ ]:
# Your turn
P = init_attn(2); m = {k: np.zeros_like(v) for k, v in P.items()}; v2 = {k: np.zeros_like(v) for k, v in P.items()}; t = 0
Ytr = np.eye(4)[ytr]
for ep in range(1, 41):
    for bi in np.array_split(rng.permutation(len(Xtr)), len(Xtr) // 64):
        p, cache = attn_fwd(P, Xtr[bi]); g = attn_bwd(P, Ytr[bi], cache, p); t += 1
        for k in P:
            m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
            P[k] -= 0.02 * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)
pte, cte = attn_fwd(P, Xte)
print(f"Transformer attention test accuracy: {np.mean(pte.argmax(1) == yte):.4f}")
print("Sample test seq:", Xte[0].tolist(), "attn weights:", cte[5][0].round(3).tolist(), "pred:", int(pte[0].argmax()), "true:", int(yte[0]))

In [ ]:
# @title Solution 4 (double-click to show the code)
P = init_attn(2); m = {k: np.zeros_like(v) for k, v in P.items()}; v2 = {k: np.zeros_like(v) for k, v in P.items()}; t = 0
Ytr = np.eye(4)[ytr]
for ep in range(1, 41):
    for bi in np.array_split(rng.permutation(len(Xtr)), len(Xtr) // 64):
        p, cache = attn_fwd(P, Xtr[bi]); g = attn_bwd(P, Ytr[bi], cache, p); t += 1
        for k in P:
            m[k] = 0.9 * m[k] + 0.1 * g[k]; v2[k] = 0.999 * v2[k] + 0.001 * g[k] ** 2
            P[k] -= 0.02 * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v2[k] / (1 - 0.999 ** t)) + 1e-8)
pte, cte = attn_fwd(P, Xte)
print(f"Transformer attention test accuracy: {np.mean(pte.argmax(1) == yte):.4f}")
print("Sample test seq:", Xte[0].tolist(), "attn weights:", cte[5][0].round(3).tolist(), "pred:", int(pte[0].argmax()), "true:", int(yte[0]))

<details><summary><b>Answer and explanation</b></summary>

The self-attention model reaches 100.0% test accuracy, while the mean-pooling baseline stalls around 50% (random guessing between the two values present in the sequence!). Inspecting the attention row shows why: the query token at position 4 learns to route weight to the matching key-value pair and read out the exact target value.

Expected output:
```
Transformer attention test accuracy: 0.5000
Sample test seq: [0, 7, 2, 4, 0] attn weights: [0.158, 0.255, 0.193, 0.249, 0.145] pred: 3 true: 3
```

</details>